# Fabric Defect Detection using ResNet CNN

This notebook implements a ResNet-based deep learning model for detecting fabric defects.

## Features:
- ResNet50 transfer learning architecture
- Data augmentation for better generalization
- Comprehensive evaluation with accuracy and confusion matrix
- Training history visualization
- Model checkpointing and early stopping

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import kagglehub

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Download Dataset from Kaggle

In [ ]:
# Download latest version of fabric defects dataset
path = kagglehub.dataset_download("nexuswho/fabric-defects-dataset")

print("Path to dataset files:", path)

# Explore dataset structure
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')

## 3. Configuration and Hyperparameters

In [ ]:
# Configuration
IMG_SIZE = (224, 224)  # ResNet50 input size
BATCH_SIZE = 32
EPOCHS = 50
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 0.001

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 4. Data Preparation and Augmentation

In [ ]:
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

# Only rescaling for validation set
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

# Create training data generator
train_generator = train_datagen.flow_from_directory(
    path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Create validation data generator
validation_generator = val_datagen.flow_from_directory(
    path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Get class names
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"\nDataset Information:")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Class distribution: {train_generator.classes}")

## 5. Visualize Sample Images

In [ ]:
# Visualize some sample images from training set
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.ravel()

# Get a batch of images
sample_images, sample_labels = next(train_generator)

for i in range(15):
    axes[i].imshow(sample_images[i])
    class_idx = np.argmax(sample_labels[i])
    axes[i].set_title(f"Class: {class_names[class_idx]}")
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Sample Fabric Images from Training Set', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 6. Build ResNet50 Model

In [ ]:
# Load pre-trained ResNet50 model without top layers
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze the base model layers
base_model.trainable = False

# Build the complete model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("ResNet50 Model Architecture:")
print("=" * 60)
model.summary()

## 7. Define Callbacks

In [ ]:
# Define callbacks for training
callbacks = [
    ModelCheckpoint(
        'best_fabric_defect_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print("- ModelCheckpoint: Saves best model based on validation accuracy")
print("- EarlyStopping: Stops training if validation loss doesn't improve for 10 epochs")
print("- ReduceLROnPlateau: Reduces learning rate when validation loss plateaus")

## 8. Train the Model

In [ ]:
print("Starting model training...")
print("=" * 60)

# Train the model
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed!")

## 9. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 0].set_title('Model Precision', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 1].set_title('Model Recall', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved as 'training_history.png'")

## 10. Model Evaluation

In [ ]:
print("Evaluating model on validation set...")
print("=" * 60)

# Reset the validation generator
validation_generator.reset()

# Get predictions
y_pred_probs = model.predict(validation_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = validation_generator.classes

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

print(f"\n{'='*60}")
print(f"MODEL EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

## 11. Confusion Matrix

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'},
    square=True,
    linewidths=0.5
)
plt.title('Confusion Matrix - Fabric Defect Detection', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix plot saved as 'confusion_matrix.png'")

# Print confusion matrix values
print("\nConfusion Matrix:")
print(cm)

## 12. Per-Class Accuracy

In [ ]:
# Calculate per-class accuracy
class_accuracies = cm.diagonal() / cm.sum(axis=1)

# Create a DataFrame for better visualization
accuracy_df = pd.DataFrame({
    'Class': class_names,
    'Accuracy': class_accuracies,
    'Percentage': class_accuracies * 100
})
accuracy_df = accuracy_df.sort_values('Accuracy', ascending=False)

print("\nPer-Class Accuracy:")
print("=" * 60)
print(accuracy_df.to_string(index=False))

# Visualize per-class accuracy
plt.figure(figsize=(12, 6))
plt.bar(accuracy_df['Class'], accuracy_df['Percentage'], color='steelblue', alpha=0.8)
plt.xlabel('Class', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Per-Class Accuracy - Fabric Defect Detection', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 100)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Save the Model

In [ ]:
# Save the final model
model.save('fabric_defect_resnet_model.h5')
print("Model saved as 'fabric_defect_resnet_model.h5'")

# Save model in SavedModel format (for production)
model.save('fabric_defect_resnet_model', save_format='tf')
print("Model also saved in TensorFlow SavedModel format as 'fabric_defect_resnet_model'")

## 14. Summary of Results

In [ ]:
print("\n" + "="*80)
print("FABRIC DEFECT DETECTION - FINAL SUMMARY")
print("="*80)
print(f"\nModel Architecture: ResNet50 (Transfer Learning)")
print(f"Input Image Size: {IMG_SIZE}")
print(f"Number of Classes: {num_classes}")
print(f"Classes: {', '.join(class_names)}")
print(f"\nTraining Samples: {train_generator.samples}")
print(f"Validation Samples: {validation_generator.samples}")
print(f"\nFinal Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nSaved Files:")
print(f"  - Model: fabric_defect_resnet_model.h5")
print(f"  - Best Model Checkpoint: best_fabric_defect_model.h5")
print(f"  - Training History Plot: training_history.png")
print(f"  - Confusion Matrix: confusion_matrix.png")
print(f"  - Per-Class Accuracy: per_class_accuracy.png")
print("\n" + "="*80)

## 15. Optional: Fine-tuning (Uncomment to run)

In [ ]:
# Uncomment the following code to perform fine-tuning

# # Unfreeze the base model
# base_model.trainable = True

# # Freeze all layers except the last 20
# for layer in base_model.layers[:-20]:
#     layer.trainable = False

# # Recompile with a lower learning rate
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=0.0001),
#     loss='categorical_crossentropy',
#     metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
# )

# print(f"Fine-tuning with {sum([1 for layer in model.layers if layer.trainable])} trainable layers")

# # Continue training
# history_fine = model.fit(
#     train_generator,
#     validation_data=validation_generator,
#     epochs=20,
#     verbose=1
# )

# # Save fine-tuned model
# model.save('fabric_defect_resnet_finetuned.h5')
# print("Fine-tuned model saved!")